# ParkCast SF — SFpark Block-Level Occupancy Features

The existing `sfpark_calibration.parquet` is a district-level anchor (10 PM districts × 24h × 2 day-types = 480 rows), which has 19.5 MAE at the block level (see `validate_sfpark_calibration.ipynb`). That loss is the district-to-block averaging, not the sensor data itself — SFpark's raw data *is* per-block.

This notebook burns through the district averaging and produces per-(blockface, hour, is_weekend) physical-occupancy features:

- `sfpark_block_occ` — sensor-derived physical occupancy for this block/hour/weekend, 2011–2013.
- `sfpark_block_coverage` — 1 if this block had SFpark sensors, 0 otherwise (most blocks didn't — SFpark was ~8 pilot districts).

Join key: `blockface_id`, derived by matching SFpark's `(STREET_NAME, BLOCK_NUM)` to SFMTA `meter_locations (street_name, street_num // 100)`.

**Output:** `data/block_sfpark_features.csv`.

## Imports

In [1]:
import os
import numpy as np
import pandas as pd

## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT_DIR, "data")

SFPARK = os.path.join(DATA_DIR, "sfpark_sensor_2011_2013.csv")
METERS = os.path.join(DATA_DIR, "meter_locations.csv")
OUT    = os.path.join(DATA_DIR, "block_sfpark_features.csv")

## `aggregate_sfpark_blocks()`

Read the sensor file once, aggregate to `(STREET_NAME, BLOCK_NUM, hour, is_weekend)`. We sum occupied-seconds and total-seconds across the entire 2011–2013 window so individual sensor outages don't skew the result — this is a long-run physical-occupancy prior, not a time-series.

In [3]:
def aggregate_sfpark_blocks():
    usecols = ["STREET_NAME", "BLOCK_NUM", "TIME_OF_DAY", "DAY_TYPE",
              "TOTAL_TIME", "TOTAL_OCCUPIED_TIME"]
    df = pd.read_csv(SFPARK, usecols=usecols)
    print(f"  SFpark rows: {len(df):,}")
    df = df.dropna(subset=["STREET_NAME", "BLOCK_NUM", "TIME_OF_DAY",
                          "DAY_TYPE", "TOTAL_TIME"])
    df = df[df["TOTAL_TIME"] > 0].copy()
    df["hour"] = (df["TIME_OF_DAY"].astype(int) // 100).astype(int)
    df["is_weekend"] = (df["DAY_TYPE"] == "weekend").astype(int)
    df["STREET_NAME"] = df["STREET_NAME"].str.strip().str.upper()
    df["BLOCK_NUM"] = df["BLOCK_NUM"].astype(int)

    agg = (df.groupby(["STREET_NAME", "BLOCK_NUM", "hour", "is_weekend"])
             [["TOTAL_TIME", "TOTAL_OCCUPIED_TIME"]].sum().reset_index())
    agg["sfpark_block_occ"] = (
        agg["TOTAL_OCCUPIED_TIME"] / agg["TOTAL_TIME"] * 100
    ).round(2)
    agg = agg[["STREET_NAME", "BLOCK_NUM", "hour", "is_weekend",
              "sfpark_block_occ"]]
    print(f"  Aggregated rows (block × hour × weekend): {len(agg):,}")
    return agg

## `map_sfpark_to_blockface()`

SFpark's (STREET_NAME, BLOCK_NUM) → SFMTA `blockface_id` via `meter_locations`. A meter at 2225 Fillmore lives on BLOCK_NUM=22 (2200 block). One SFpark block typically corresponds to one blockface on each side of the street — we take the majority blockface_id per (street, block_num) group.

In [4]:
def map_sfpark_to_blockface():
    m = pd.read_csv(METERS, usecols=["street_name", "street_num", "blockface_id"])
    m = m.dropna(subset=["street_name", "street_num", "blockface_id"])
    m["STREET_NAME"] = m["street_name"].str.strip().str.upper()
    m["BLOCK_NUM"] = (m["street_num"].astype(float) // 100).astype(int)
    # Take the most common blockface on each (street, block_num). Meters on
    # opposite sides of the same block share STREET_NAME+BLOCK_NUM but have
    # different blockface_ids — the one with more meters wins. A block-level
    # SFpark signal is applied to both faces either way (both share demand).
    mapping = (m.groupby(["STREET_NAME", "BLOCK_NUM", "blockface_id"])
                .size().reset_index(name="n"))
    mapping = mapping.sort_values(["STREET_NAME", "BLOCK_NUM", "n"],
                                ascending=[True, True, False])
    mapping = mapping.drop_duplicates(["STREET_NAME", "BLOCK_NUM"]).drop(columns="n")
    print(f"  Unique SFMTA (street, block_num) → blockface entries: {len(mapping):,}")
    return mapping

## Pipeline

In [5]:
print("=" * 60)
print("ParkCast SF — SFpark Block-Level Features")
print("=" * 60)

agg = aggregate_sfpark_blocks()
print(f"  Unique SFpark blocks: {agg[['STREET_NAME','BLOCK_NUM']].drop_duplicates().shape[0]:,}")

mapping = map_sfpark_to_blockface()

joined = agg.merge(mapping, on=["STREET_NAME", "BLOCK_NUM"], how="inner")
print(f"  Matched rows after join: {len(joined):,}")
matched_blocks = joined["blockface_id"].nunique()
print(f"  Matched unique blockfaces: {matched_blocks:,}")

feats = (joined.groupby(["blockface_id", "hour", "is_weekend"])
               ["sfpark_block_occ"].mean().reset_index())
feats["sfpark_block_coverage"] = 1
feats = feats.rename(columns={"hour": "hour_of_day"})
feats.to_csv(OUT, index=False)

print(f"\nFeature rows (blockface × hour × weekend): {len(feats):,}")
print("  Summary:")
print(feats["sfpark_block_occ"].describe().round(2).to_string())
print(f"\nSaved → {OUT}")
print("=" * 60)

ParkCast SF — SFpark Block-Level Features


  SFpark rows: 7,902,291


  Aggregated rows (block × hour × weekend): 19,488
  Unique SFpark blocks: 406
  Unique SFMTA (street, block_num) → blockface entries: 1,714
  Matched rows after join: 19,296
  Matched unique blockfaces: 402

Feature rows (blockface × hour × weekend): 19,296
  Summary:
count    19296.00
mean        51.27
std         21.53
min          0.00
25%         34.06
50%         52.93
75%         69.55
max         93.48

Saved → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/block_sfpark_features.csv
